# ComfyUI + FreeField on Kaggle
Run cells top to bottom. Turn **Internet ON** in the notebook settings first.

## 1. Install ComfyUI

In [ ]:
%cd /kaggle/working
!git clone https://github.com/comfyanonymous/ComfyUI.git comfyui
%cd comfyui
!pip install -q -r requirements.txt

## 2. Store Models in /tmp
Models go to `/tmp` so they don't fill up `/kaggle/working`.

In [ ]:
import os, shutil

shutil.rmtree("/kaggle/working/comfyui/models", ignore_errors=True)
for d in ["checkpoints", "diffusion_models", "text_encoders", "vae", "loras",
          "clip_vision", "upscale_models", "controlnet", "clip", "unet",
          "embeddings", "vae_approx", "latent_upscale_models"]:
    os.makedirs(f"/tmp/models/{d}", exist_ok=True)
os.symlink("/tmp/models", "/kaggle/working/comfyui/models")

!ls -la /kaggle/working/comfyui | grep models

## 3. Install FreeField Node

In [ ]:
import os

%cd /kaggle/working/comfyui/custom_nodes
!git clone https://github.com/neuregex/ComfyUI-freefield
if os.path.exists("ComfyUI-freefield/requirements.txt"):
    !pip install -q -r ComfyUI-freefield/requirements.txt

## 4. Install Cloudflared
Used to get a public link to the ComfyUI UI.

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

## 5. Set HF Token
Paste your Hugging Face token below.

In [ ]:
import os
os.environ["HF_TOKEN"] = "hf_your_token_here"   # paste your real token
print("HF_TOKEN set")

## 6. Start ComfyUI
Waits until the server is up. If it fails, the log is printed.

In [ ]:
%cd /kaggle/working/comfyui
import subprocess, time, urllib.request

!pkill -f "main.py" || true

subprocess.Popen(
    'python main.py --listen 0.0.0.0 --port 8188 --preview-method auto --lowvram '
    '--enable-cors-header "*" > /tmp/comfy.log 2>&1',
    shell=True
)

for i in range(120):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188", timeout=1)
        print(f"ComfyUI ready after ~{i}s")
        break
    except Exception:
        time.sleep(1)
else:
    print("Not ready, check the log:")
    !tail -n 30 /tmp/comfy.log

## 7. Get Public Link
Open the `https://...trycloudflare.com` link printed below. Keep this cell running.

In [ ]:
!cloudflared tunnel --url http://127.0.0.1:8188

## 8. Stop Everything
Run this when you're done.

In [ ]:
!pkill -f "[m]ain.py"; pkill -f "[c]loudflared"